# Comprehensive Evaluation & Model Comparison

## Objective
Conduct thorough evaluation of all forecasting models:
1. Compare all models (baselines + advanced)
2. Visualize predictions and errors
3. Perform error diagnostics
4. Analyze where models succeed and fail
5. Select best model with justification

## Evaluation Framework
- **Quantitative Metrics**: MAE, RMSE, MAPE
- **Visual Analysis**: Prediction plots, error distributions
- **Error Patterns**: By day of week, over time
- **Model Interpretability**: Feature importance, behavior analysis

In [1]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

# Import project modules
import src.config as config
from src.utils import print_section_header
from src.evaluation import (
    compute_metrics,
    plot_predictions,
    plot_errors,
    plot_residuals,
    create_comparison_table,
    error_diagnostics,
    plot_error_by_day_of_week,
    plot_scatter_actual_vs_predicted,
    compare_multiple_models
)

# Set random seed
np.random.seed(config.RANDOM_SEED)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Environment setup complete!")

Environment setup complete!


## 1. Load All Results

In [2]:
# Load test data
y_test = np.load(config.DATA_PATH / 'y_test.npy')
test_dates = np.load(config.DATA_PATH / 'test_dates.npy', allow_pickle=True)

# Load all model results
with open(config.DATA_PATH / 'all_model_results.pkl', 'rb') as f:
    all_results = pickle.load(f)

with open(config.DATA_PATH / 'baseline_results.pkl', 'rb') as f:
    baseline_results = pickle.load(f)

with open(config.DATA_PATH / 'main_model_results.pkl', 'rb') as f:
    main_results = pickle.load(f)

print("All results loaded successfully!")
print(f"Test set size: {len(y_test)}")
print(f"Models evaluated: {len(all_results['all_metrics'])}")

All results loaded successfully!
Test set size: 254
Models evaluated: 8


## 2. Comprehensive Model Comparison Table

In [3]:
# Display complete comparison table
print_section_header("Complete Model Performance Comparison")
results_df = all_results['results_df']
print(results_df)

print(f"\n{'='*60}")
print(f"✓ Best Model: {all_results['best_model']}")
print(f"  MAE:  {results_df.loc[all_results['best_model'], 'MAE']:.2f}")
print(f"  RMSE: {results_df.loc[all_results['best_model'], 'RMSE']:.2f}")
print(f"{'='*60}")


 Complete Model Performance Comparison

                                MAE         RMSE
RandomForest            1488.616848  2508.228777
Linear Regression       1669.273479  2556.153372
GradientBoosting        1671.046594  2512.275550
SARIMAX                 2278.688946  2987.621296
SARIMA                  2769.459164  3440.913312
Moving Average (7-day)  3416.858830  3924.052913
Seasonal Naive (7-day)  8466.059055  9175.241916
Naive                   8565.224409  9220.623933

✓ Best Model: RandomForest
  MAE:  1488.62
  RMSE: 2508.23


In [ ]:
# Visualize performance comparison
fig, ax = plt.subplots(figsize=(12, 8))

results_df.plot(kind='barh', ax=ax, alpha=0.8, edgecolor='black', width=0.75)
ax.set_xlabel('Error', fontsize=13)
ax.set_ylabel('Model', fontsize=13)
ax.set_title('Model Performance Comparison (All Models)', fontsize=15, fontweight='bold')
ax.legend(title='Metric', fontsize=11, title_fontsize=12)
ax.grid(True, alpha=0.3, axis='x')
ax.invert_yaxis()

# Highlight best model
best_idx = list(results_df.index).index(all_results['best_model'])
ax.get_children()[best_idx].set_color('gold')
ax.get_children()[best_idx + len(results_df)].set_color('orange')

plt.tight_layout()

from src.utils import save_figure
save_figure(fig, '20_final_model_comparison.png')
plt.show()

## 3. Baseline vs Advanced Models Analysis

In [5]:
# Separate baselines from advanced models
baseline_models = ['Naive', 'Seasonal Naive (7-day)', 'Moving Average (7-day)', 'Linear Regression']
advanced_models = ['SARIMA', 'SARIMAX', 'RandomForest', 'GradientBoosting']

baseline_mae = results_df.loc[baseline_models, 'MAE']
advanced_mae = results_df.loc[advanced_models, 'MAE']

print_section_header("Baseline vs Advanced Models")
print(f"Best Baseline:  {baseline_mae.idxmin()} (MAE: {baseline_mae.min():.2f})")
print(f"Best Advanced:  {advanced_mae.idxmin()} (MAE: {advanced_mae.min():.2f})")
print(f"\nImprovement: {((baseline_mae.min() - advanced_mae.min()) / baseline_mae.min() * 100):.2f}%")


 Baseline vs Advanced Models

Best Baseline:  Linear Regression (MAE: 1669.27)
Best Advanced:  RandomForest (MAE: 1488.62)

Improvement: 10.82%


## 4. Prediction Visualization for Top Models

In [6]:
# Get top 3 models
top_3_models = results_df.index[:3]

print(f"Top 3 Models:")
for i, model in enumerate(top_3_models, 1):
    print(f"{i}. {model} (MAE: {results_df.loc[model, 'MAE']:.2f})")

Top 3 Models:
1. RandomForest (MAE: 1488.62)
2. Linear Regression (MAE: 1669.27)
3. GradientBoosting (MAE: 1671.05)


In [ ]:
# Collect predictions for top models
all_predictions = {}

# Baseline predictions
all_predictions.update({
    'Naive': baseline_results['predictions']['naive'],
    'Seasonal Naive (7-day)': baseline_results['predictions']['seasonal_naive'],
    'Moving Average (7-day)': baseline_results['predictions']['moving_average'],
    'Linear Regression': baseline_results['predictions']['linear_regression']
})

# Advanced model predictions
all_predictions.update({
    'SARIMA': main_results['predictions']['sarima'],
    'SARIMAX': main_results['predictions']['sarimax'],
    'RandomForest': main_results['predictions']['random_forest'],
    'GradientBoosting': main_results['predictions']['gradient_boosting']
})

# Plot top 3 models
top_3_predictions = {model: all_predictions[model] for model in top_3_models}

fig = compare_multiple_models(
    y_test,
    top_3_predictions,
    test_dates,
    save_filename='21_top_3_models_comparison.png'
)
plt.show()

## 5. Best Model Detailed Analysis

In [8]:
# Get best model predictions
best_model_name = all_results['best_model']
best_predictions = all_predictions[best_model_name]

print_section_header(f"Best Model: {best_model_name}")

# Compute detailed metrics
best_metrics = compute_metrics(y_test, best_predictions)
print(f"MAE:  {best_metrics['MAE']:.2f}")
print(f"RMSE: {best_metrics['RMSE']:.2f}")
if not np.isnan(best_metrics['MAPE']):
    print(f"MAPE: {best_metrics['MAPE']:.2f}%")


 Best Model: RandomForest

MAE:  1488.62
RMSE: 2508.23


In [ ]:
# Plot predictions
fig = plot_predictions(
    y_test,
    best_predictions,
    test_dates,
    title=f'Best Model Predictions: {best_model_name}',
    save_filename='22_best_model_predictions.png'
)
plt.show()

In [ ]:
# Plot errors over time
fig = plot_errors(
    y_test,
    best_predictions,
    test_dates,
    title=f'Prediction Errors: {best_model_name}',
    save_filename='23_best_model_errors.png'
)
plt.show()

In [ ]:
# Scatter plot: Actual vs Predicted
fig = plot_scatter_actual_vs_predicted(
    y_test,
    best_predictions,
    title=f'Actual vs Predicted: {best_model_name}',
    save_filename='24_best_model_scatter.png'
)
plt.show()

## 6. Residual Analysis

In [ ]:
# Calculate residuals
best_residuals = y_test - best_predictions

# Plot residual distribution
fig = plot_residuals(
    best_residuals,
    title=f'Residual Analysis: {best_model_name}',
    save_filename='25_best_model_residuals.png'
)
plt.show()

# Print residual statistics
print_section_header("Residual Statistics")
print(f"Mean:     {np.mean(best_residuals):.2f}")
print(f"Std Dev:  {np.std(best_residuals):.2f}")
print(f"Min:      {np.min(best_residuals):.2f}")
print(f"Max:      {np.max(best_residuals):.2f}")
print(f"\nShould be centered at 0 with normal distribution for well-performing model.")

## 7. Error Patterns by Day of Week

In [ ]:
# Analyze errors by day of week
fig = plot_error_by_day_of_week(
    y_test,
    best_predictions,
    test_dates,
    save_filename='26_errors_by_day_of_week.png'
)
plt.show()

print("\nInterpretation:")
print("- Mean error shows systematic over/under-prediction by day")
print("- MAE shows which days are hardest to predict")
print("- Look for patterns related to weekly seasonality")

## 8. Error Diagnostics: Identifying Large Errors

In [ ]:
# Identify large errors (95th percentile)
error_analysis = error_diagnostics(
    y_test,
    best_predictions,
    test_dates,
    threshold_percentile=95
)

print_section_header("Large Error Events (Top 5%)")
print(f"Total large error days: {len(error_analysis)}")
print(f"\nTop 10 largest errors:")
print(error_analysis.head(10))

In [15]:
# Analyze large errors
print("\nLarge Error Analysis:")
print(f"Mean large error: {error_analysis['abs_error'].mean():.2f}")
print(f"\nDays with largest errors might correspond to:")
print("  - Holidays or special events")
print("  - Unexpected promotions")
print("  - Stock-outs or supply chain issues")
print("  - Data anomalies")


Large Error Analysis:
Mean large error: 7921.11

Days with largest errors might correspond to:
  - Holidays or special events
  - Unexpected promotions
  - Stock-outs or supply chain issues
  - Data anomalies


## 9. Feature Importance Analysis (for ML models)

In [ ]:
# If best model is ML-based, show feature importance
if best_model_name in ['RandomForest', 'GradientBoosting']:
    print_section_header(f"Feature Importance: {best_model_name}")
    
    model_key = 'random_forest' if best_model_name == 'RandomForest' else 'gradient_boosting'
    importance_df = main_results['feature_importance'][model_key]
    
    print("\nTop 15 Most Important Features:")
    print(importance_df.head(15))
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 8))
    importance_df.head(15).plot(x='feature', y='importance', kind='barh', ax=ax,
                                color='teal', alpha=0.8, edgecolor='black', legend=False)
    ax.set_xlabel('Importance', fontsize=12)
    ax.set_ylabel('Feature', fontsize=12)
    ax.set_title(f'{best_model_name}: Feature Importance', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_yaxis()
    
    plt.tight_layout()
    save_figure(fig, '27_best_model_feature_importance.png')
    plt.show()
else:
    print(f"\n{best_model_name} is a statistical model - feature importance not applicable.")

## 10. Model Selection Justification

In [17]:
print_section_header("Model Selection Justification")

print(f"Selected Model: {best_model_name}")
print(f"\nPerformance Metrics:")
print(f"  MAE:  {results_df.loc[best_model_name, 'MAE']:.2f}")
print(f"  RMSE: {results_df.loc[best_model_name, 'RMSE']:.2f}")

print(f"\nComparison to Baselines:")
best_baseline_mae = baseline_mae.min()
improvement = (best_baseline_mae - results_df.loc[best_model_name, 'MAE']) / best_baseline_mae * 100
print(f"  Best Baseline MAE: {best_baseline_mae:.2f}")
print(f"  Improvement: {improvement:.2f}%")

print(f"\nStrengths:")
if best_model_name in ['SARIMA', 'SARIMAX']:
    print("  - Captures trend and seasonality statistically")
    print("  - Interpretable coefficients")
    print("  - Well-suited for univariate time series")
    if best_model_name == 'SARIMAX':
        print("  - Incorporates exogenous variables (promotions)")
elif best_model_name in ['RandomForest', 'GradientBoosting']:
    print("  - Captures non-linear patterns")
    print("  - Handles multiple features effectively")
    print("  - Robust to outliers")
    print("  - Provides feature importance insights")
else:
    print("  - Simple and interpretable")
    print("  - Fast to train and predict")

print(f"\nLimitations:")
print("  - Trained on single store/family (scope limitation)")
print("  - Performance may vary with different products")
print("  - Large errors on some days (holidays, special events)")

print(f"\nRecommendation:")
print(f"  ✓ {best_model_name} is recommended for this sales forecasting task")
print(f"  ✓ Achieves best balance of accuracy and interpretability")
print(f"  ✓ Suitable for production deployment with monitoring")


 Model Selection Justification

Selected Model: RandomForest

Performance Metrics:
  MAE:  1488.62
  RMSE: 2508.23

Comparison to Baselines:
  Best Baseline MAE: 1669.27
  Improvement: 10.82%

Strengths:
  - Captures non-linear patterns
  - Handles multiple features effectively
  - Robust to outliers
  - Provides feature importance insights

Limitations:
  - Trained on single store/family (scope limitation)
  - Performance may vary with different products
  - Large errors on some days (holidays, special events)

Recommendation:
  ✓ RandomForest is recommended for this sales forecasting task
  ✓ Achieves best balance of accuracy and interpretability
  ✓ Suitable for production deployment with monitoring


## 11. Summary

### Evaluation Results:

**Models Evaluated:**
- 4 Baseline models
- 4 Advanced models
- Total: 8 forecasting approaches

**Best Performing Model:**
- [Will be determined by results]
- MAE: [value]
- RMSE: [value]
- Improvement over baseline: [percentage]

**Key Insights:**
1. Advanced models outperform simple baselines
2. Feature engineering significantly impacts ML model performance
3. [Statistical/ML] models better suited for this problem
4. Weekly seasonality is the strongest pattern
5. Some days remain difficult to predict (special events)

**Error Patterns:**
- Residuals approximately normally distributed
- Some systematic errors by day of week
- Large errors on [specific days/events]

**Model Interpretability:**
- Most important features: [from feature importance]
- Lag features capture recent trends
- Calendar features capture seasonality
- Promotions have measurable impact

### Next Steps:
- Apply best model to business use case (inventory planning)
- Generate 30-day forecast
- Calculate inventory recommendations
- Document findings and recommendations

In [18]:
# Save evaluation summary
evaluation_summary = {
    'best_model': best_model_name,
    'best_predictions': best_predictions,
    'best_metrics': best_metrics,
    'all_predictions': all_predictions,
    'results_table': results_df,
    'large_errors': error_analysis
}

with open(config.DATA_PATH / 'evaluation_summary.pkl', 'wb') as f:
    pickle.dump(evaluation_summary, f)

print("\n✓ Evaluation summary saved!")


✓ Evaluation summary saved!
